# Modality hopping demonstration with ShEPhERD-2 (peptide -> small molecule)

This tutorial demonstrates "modality hopping": conditioning generation on the interaction profile of one molecular modality (a peptide) to design a different modality (a small molecule) that mimics its key interactions. We cover these use-cases:
- Extracting an interaction profile directly from a peptide PDB structure
- Subselecting the profile to the peptide-receptor interface
- Pharmacophore-conditioned generation of a small molecule
- Extracting and visualizing the generated small molecule against the reference peptide


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import pickle
import time

import numpy as np
import matplotlib.pyplot as plt

import rdkit
from rdkit import Chem
from rdkit.Chem import AllChem

import torch
# faster inference
torch.set_float32_matmul_precision('high')

from shepherd_score.visualize import draw_sample, view_sample_trajectory, draw_2d_highlight, draw
from shepherd_score.container import Molecule
from shepherd_score.evaluations.evaluate.pipelines import ConditionalEvalPipeline

from shepherd import load_model
from shepherd.interaction_profile import extract_interaction_profile, ConditionAtoms, InpaintAdvancedOptions
from shepherd.extract import create_rdkit_molecule
import shepherd.comp_inference.utils as utils


## 1. Load the model
Downloads from Huggingface automatically.


In [ ]:
model = load_model(
    model_type='mosesaq',
    inference_only=True,
    device='cuda'
)
params = model.params


## 2. Extract and subselect peptide interaction profile

Unlike the small-molecule tutorials, here the reference comes directly from a peptide PDB structure rather than a SMILES string, so we set `xtb_optimize=False` and instead rely on the profile derived straight from the crystal/docked pose. We then subselect the full peptide profile down to just the atoms and pharmacophores at the peptide-receptor interface — this is the region ShEPhERD-2 will try to mimic when it designs a small molecule.


In [ ]:
mol1_path = '../data/conformers/modality_hopping/hcmv_8j3s_peptide.pdb'
mol_1 = Chem.MolFromPDBFile(mol1_path, removeHs=False)
profile = extract_interaction_profile(
    mol_1,
    xtb_optimize=False
)
print(f"n_atoms:       {profile.n_atoms}")
print(f"n_pharms:      {profile.n_pharms}")


In [15]:
# Subselection of profile - in this case decided based on peptide interface with target receptor
pharm_atoms_to_keep = [96, 71, 51, 44, 30, 20, 24, 23, 87]
surf_atoms_to_keep = [20, 24, 21, 22, 23, 27, 28, 29, 30, 41, 42, 43, 48, 96, 71, 51, 44, 87, 91, 49, 50, 65, 66, 67, 86, 2]
sub_profile = profile.subselection(pharm_atoms_to_keep, surf_atoms_to_keep, include_h = False, radial_buffer=1.5)
print(f"n_atoms:       {sub_profile.n_atoms}")
print(f"n_pharms:      {sub_profile.n_pharms}")


n_atoms:       206
n_pharms:      12


In [13]:
view_sub = draw(
    mol=sub_profile.mol,
    pharm_types=sub_profile.pharm_types,
    pharm_ancs=sub_profile.pharm_positions,
    pharm_vecs=sub_profile.pharm_directions,
    point_cloud=sub_profile.surface,
    esp=sub_profile.electrostatics,
    opacity=0.5,
    width=800,
    height=500,
)
view_sub.show()

3Dmol.js failed to load for some reason. Please check your browser console for error messages.

## 3. Modality hopping generation

We generate small-molecule samples conditioned only on the pharmacophores of the peptide interface profile (`pharmacophore_conditioning=True`), which is what lets the model "hop" from the peptide modality to a small molecule rather than trying to reconstruct the peptide itself. `N_x4` gives a few extra pharmacophore slots beyond `profile.n_pharms` for inpainting flexibility, and `N_x1` sets the number of atoms in the generated molecule (independent of the peptide's atom count). We also disable ESP inpainting (`InpaintAdvancedOptions(inpaint_x3_x=False)`) since the peptide's surface electrostatics aren't a reliable target for a small molecule.


In [ ]:
samples = model.generate(
    batch_size=20, # tune based on GPU
    N_x1=75, # number of atoms in the generated small molecule
    N_x4=profile.n_pharms + 12, # number of pharmacophores, with extra slots for inpainting flexibility
    condition=sub_profile,
    pharmacophore_conditioning=True,
    inpaint=InpaintAdvancedOptions(
        inpaint_x3_x=False, # don't inpaint ESP values
    ),
)


## 4. Extraction / processing: samples -> RDKit mols -> SMILES

As in the other tutorials, we can convert generated samples to RDKit `Mol` objects in two ways:
- A quick conversion via `create_rdkit_molecule`, followed by an MMFF optimization — fast, but does not recompute partial charges.
- A xTB-relaxed conversion via `GeneratedSample.to_mol_charges()`, which also returns the xTB partial charges for each generated molecule. This is slower but gives higher-quality structures and charges.


In [30]:
# Quick conversion, no xTB relaxation 
quick_mols = []
for sample in samples:
    mol = create_rdkit_molecule(sample)
    if mol is not None:
        mol = Chem.RemoveHs(mol)
        AllChem.MMFFOptimizeMolecule(mol)
    quick_mols.append(mol)

quick_smiles = [Chem.MolToSmiles(m) if m is not None else None for m in quick_mols]
print(quick_smiles)

[14:52:20] Molecule does not have explicit Hs. Consider calling AddHs()
[14:52:20] Molecule does not have explicit Hs. Consider calling AddHs()
[14:52:24] Molecule does not have explicit Hs. Consider calling AddHs()
[14:52:24] Molecule does not have explicit Hs. Consider calling AddHs()
[14:52:24] Molecule does not have explicit Hs. Consider calling AddHs()
[14:52:31] Molecule does not have explicit Hs. Consider calling AddHs()
2026-09-03 14:52:31,349 - root - WARNING - Molecule is a fragment
[14:52:31] Molecule does not have explicit Hs. Consider calling AddHs()
[14:52:31] Molecule does not have explicit Hs. Consider calling AddHs()
[14:53:28] Molecule does not have explicit Hs. Consider calling AddHs()
2026-09-03 14:53:29,735 - root - WARNING - Molecule is a fragment
[14:53:29] Molecule does not have explicit Hs. Consider calling AddHs()
[14:53:33] Molecule does not have explicit Hs. Consider calling AddHs()
[14:53:34] Molecule does not have explicit Hs. Consider calling AddHs()
[14:

['CC1=Nn2nc(CCCOc3ccsc3)cc2-c2sc(-c3c(NCC4CC4)ncnc3C(=O)Nc3nccnc3O)nc21', 'C#Cc1nc(C2=C/C(=C3/N4[C-](CNC5=NC67NC6N7C=N5)[S+]34)[N-]C(C3=[N+]4O[C@@]34[C@H](C)[C@@H](C)c3n[nH]c(C(=O)CC)c3C)=C2)n[nH]1', 'CCOC1=[N+]=Cc2c(cnnc2CCCOc2cc(CNC3=Nn4nc(SCC5=CC=C[CH-]C=N5)[s+]c4[N-]3)nc(OC)n2)N=N1', 'CN[C@H](C1=NC(C)=NN=C2C=C(COCc3cc4ccccc4cn3)N=[N+]21)c1nnn(Cc2nnc(C)c(C)n2)n1', 'CCN1Cc2nc(OC)nc(n2)-n2c(SCCCCc3nccnn3)nc3nc(C)nc(c32)Cc2nc(c(Br)s2)[C@H]1C', 'C=Cc1nnc(-c2nc(N[C@@H](C)CCc3ncc4nc(/[NH+]=C/C5=NNC(=O)C=N5)nnc4n3)c3c(C)nn(C)c3n2)c(N)n1', None, 'COC[C@@H](COc1nc2[nH]nc(C)c2c2c1NC(=O)C(Cc1nc(CCc3cccnc3)no1)=N2)Cc1nnc(C)o1', 'Cc1nc(CO)c2[o+]c3ccc(C#N)nc3nc2c1-c1ncnc(CCCCCCc2noc(Cc3ccccn3)n2)n1', 'Cc1c2nc3n1N=C(S3)[C@H](C)C/[C-]=N/[N+]#CC1=[S+]/C(=C\\O)C(=C1)c1nc(NCc3nc(-c4ccsn4)no3)n(n1)Cc1nc(N)c([nH]1)C2=O', None, 'CCCCC1=NC(C)=CC2=[N+](C3CC3)C(=O)N(Cc3cc4nnc(-c5nc(-c6ccccc6)n[nH]5)cc4nc3C#N)C2=N1', 'CCNc1nc(C)c(=O)c2c3c(c(CC)nnc13)N1C3=NC(c4noc(NCc5cnccn5)n4)=NC4=C5NC(=O)[S+]=C5C(=C21)N34'

In [ ]:
# xTB-relaxed conversion + partial charges
relaxed_mols, relaxed_charges = [], []
for sample in samples:
    mol, charges = sample.to_mol_charges(xtb_optimize=True, xtb_timeout=180)
    relaxed_mols.append(mol)
    relaxed_charges.append(charges)

relaxed_smiles = [
    Chem.MolToSmiles(Chem.RemoveHs(m)) if m is not None else None
    for m in relaxed_mols
]
print(relaxed_smiles)


## 5. Visualize generated small molecule sample against reference peptide

We overlay a generated small molecule against the reference peptide interface to see how well it recapitulates the peptide's key interactions.


In [45]:
sample_ind = 8
draw_sample(generated_sample=samples[sample_ind], ref_mol=sub_profile.mol, only_atoms=True)


3Dmol.js failed to load for some reason. Please check your browser console for error messages.